In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

%md
## Ingestion del archivo country.json

In [0]:
country_df = (
    spark.read
        .format('json')
        .option('inferSchema', True)
        .load(f'{bronze_folder_path}/country.json')
)
display(country_df)
country_df.printSchema()

countryId,countryIsoCode,countryName
128,AE,United Arab Emirates
129,AF,Afghanistan
130,AO,Angola
131,AR,Argentina
132,AT,Austria
133,AU,Australia
134,AW,Aruba
135,BA,Bosnia and Herzegovina
136,BE,Belgium
137,BG,Bulgaria


root
 |-- countryId: long (nullable = true)
 |-- countryIsoCode: string (nullable = true)
 |-- countryName: string (nullable = true)



In [0]:
country_schema = "countryId string, countryIsoCode string, countryName string"
country_df = (
    spark.read
        .format('json')
        .option('header', True)
        .schema(country_schema)
        .load(f'{bronze_folder_path}/country.json')
)
display(country_df)
country_df.printSchema()

countryId,countryIsoCode,countryName
128,AE,United Arab Emirates
129,AF,Afghanistan
130,AO,Angola
131,AR,Argentina
132,AT,Austria
133,AU,Australia
134,AW,Aruba
135,BA,Bosnia and Herzegovina
136,BE,Belgium
137,BG,Bulgaria


root
 |-- countryId: string (nullable = true)
 |-- countryIsoCode: string (nullable = true)
 |-- countryName: string (nullable = true)



### Eliminar columnas

#### Metodo 1

In [0]:
country_df_dropped = country_df.drop('countryIsoCode')

display(country_df_dropped)

countryId,countryName
128,United Arab Emirates
129,Afghanistan
130,Angola
131,Argentina
132,Austria
133,Australia
134,Aruba
135,Bosnia and Herzegovina
136,Belgium
137,Bulgaria


#### Metodo 2

In [0]:
country_df_dropped = country_df.drop(country_df['countryIsoCode'])

display(country_df_dropped)

countryId,countryName
128,United Arab Emirates
129,Afghanistan
130,Angola
131,Argentina
132,Austria
133,Australia
134,Aruba
135,Bosnia and Herzegovina
136,Belgium
137,Bulgaria


#### Metodo 3

In [0]:
from pyspark.sql.functions import col, lit, current_timestamp

In [0]:
country_df_dropped = country_df.drop(col('countryIsoCode'))

display(country_df_dropped)

countryId,countryName
128,United Arab Emirates
129,Afghanistan
130,Angola
131,Argentina
132,Austria
133,Australia
134,Aruba
135,Bosnia and Herzegovina
136,Belgium
137,Bulgaria


In [0]:
country_df_final = (
    add_ingestion_date(country_df_dropped)
        .withColumnsRenamed({'countryId': 'country_id', 'countryName': 'country_name'})
        .withColumn('env', lit(v_env))
)
display(country_df_final)

country_id,country_name,ingestion_date,env
128,United Arab Emirates,2026-10-05T06:43:27.651Z,dev
129,Afghanistan,2026-10-05T06:43:27.651Z,dev
130,Angola,2026-10-05T06:43:27.651Z,dev
131,Argentina,2026-10-05T06:43:27.651Z,dev
132,Austria,2026-10-05T06:43:27.651Z,dev
133,Australia,2026-10-05T06:43:27.651Z,dev
134,Aruba,2026-10-05T06:43:27.651Z,dev
135,Bosnia and Herzegovina,2026-10-05T06:43:27.651Z,dev
136,Belgium,2026-10-05T06:43:27.651Z,dev
137,Bulgaria,2026-10-05T06:43:27.651Z,dev


In [0]:
(
    country_df_final.write
        .mode('overwrite')
        .format('parquet')
        .save(f'{silver_folder_path}/countries')
)

In [0]:
display(dbutils.fs.ls(f'{silver_folder_path}/countries'))

path,name,size,modificationTime
abfss://silver@moviehistoryproject.dfs.core.windows.net/countries/_SUCCESS,_SUCCESS,0,1791182609000
abfss://silver@moviehistoryproject.dfs.core.windows.net/countries/_committed_9036741917301868325,_committed_9036741917301868325,124,1791182609000
abfss://silver@moviehistoryproject.dfs.core.windows.net/countries/_started_9036741917301868325,_started_9036741917301868325,0,1791182609000
abfss://silver@moviehistoryproject.dfs.core.windows.net/countries/part-00000-tid-9036741917301868325-5b56b450-8557-419d-934d-b2d5f11692a1-638-1.c000.snappy.parquet,part-00000-tid-9036741917301868325-5b56b450-8557-419d-934d-b2d5f11692a1-638-1.c000.snappy.parquet,2849,1791182609000


In [0]:
dbutils.notebook.exit("Success")